# Day 17 (Part 2): Preparing Data and Training the Model

Part 1 ([`Notebook.ipynb`](Notebook.ipynb)) built `CustomerPredictor`, a small network with 3 inputs and 1 output. Before it can learn, it needs data. This notebook prepares that data step by step, then trains the model on it and keeps the version that does best on the validation set.

![Part 2 roadmap: prepare the data (create, split, scale, convert to tensors, mini-batches), then build and train the model (model, loss and optimiser, training), and next evaluate it on the test set](images/data_pipeline.svg)

### How to use this notebook
- Run the cells **from top to bottom**. Each step uses the variables made by the step before it (`features`, `train_features`, `train_features_scaled`, ...).
- Each section follows the same pattern: **the idea → the code → what the output means**.
- The diagrams are in the `images/` folder.

### Contents

| Section | What you will learn |
|---|---|
| 1. Importing the Libraries | Which library does which job |
| 2. Creating the Dataset | A practice dataset of 1000 customers with `make_classification` |
| 3. Splitting into Training, Validation and Test Sets | Why the data is split three ways |
| 4. Feature Scaling with `StandardScaler` | Scaling, and `fit_transform` vs `transform` |
| 5. Converting the Data to PyTorch Tensors | `float32` and the `(n, 1)` label shape |
| 6. Mini-batches with `TensorDataset` and `DataLoader` | Batches, iterations, epochs and shuffling |
| 7. Creating the Model | The same `CustomerPredictor` as Part 1 |
| 8. The Loss Function: Binary Cross-Entropy (BCE) | How the model's mistakes become one number |
| 9. Training the Model | The training loop with batch and epoch logs, validation, and saving the best model |

## 1. Importing the Libraries

All the tools for this notebook are imported once, at the top. Each one has one job:

| Import | Job | Used in |
|---|---|---|
| `torch`, `torch.nn` | Tensors, layers and loss functions | Sections 5, 7, 8 |
| `torch.optim` | Optimisers that update the weights (SGD, Adam) | Training |
| `TensorDataset`, `DataLoader` | Cut the data into mini-batches | Section 6 |
| `make_classification` | Creates a practice dataset | Section 2 |
| `train_test_split` | Splits the data into training, validation and test sets | Section 3 |
| `StandardScaler` | Puts every feature on the same scale | Section 4 |
| `accuracy_score`, `classification_report`, `confusion_matrix` | Measure how good the predictions are | Evaluation |
| `matplotlib.pyplot` | Draws charts, such as the training loss | Training |

> `torch.optim`, the metrics and `plt` are not used yet. They are needed later, when the model is trained and evaluated.

In [ ]:
# PyTorch: tensors, layers, optimisers and data loading
import torch
import torch.nn as nn  # layers (nn.Linear) and loss functions
import torch.optim as optim  # optimisers that update the weights (e.g. SGD, Adam)
from torch.utils.data import (
    DataLoader,
    TensorDataset,
)  # feed the data to the model in mini-batches

# scikit-learn: create, split, scale and score the data
from sklearn.datasets import (
    make_classification,
)  # generates a fake classification dataset
from sklearn.model_selection import (
    train_test_split,
)  # splits the data into training and test sets
from sklearn.preprocessing import (
    StandardScaler,
)  # rescales each feature to mean 0, standard deviation 1

# measure how good the predictions are
from sklearn.metrics import accuracy_score, classification_report, confusion_matrix

# Matplotlib: draw charts (e.g. the training loss)
import matplotlib.pyplot as plt

## 2. Creating the Dataset

Instead of loading a real file, `make_classification` from scikit-learn **generates** a practice dataset. Here it simulates 1000 customers:

| Variable | Shape | What it holds |
|---|---|---|
| `features` | `(1000, 3)` | The **features**: 3 numbers that describe each customer, one row per customer |
| `labels` | `(1000,)` | The **labels**: the answer to predict for each customer, `0` or `1` (for example, "will not buy" or "will buy") |

- `n_features=3` matches the first layer of `CustomerPredictor` from Part 1, `nn.Linear(3, 8)`.
- The two classes are almost balanced: 501 zeros and 499 ones.
- `random_state=42` makes scikit-learn create the same data on every run, so your numbers match these notes.

In a real project, this step is usually loading a file, for example with `pd.read_csv(...)`. The rest of the pipeline stays the same.

In [ ]:
# Fix PyTorch's random seed so the model starts with the same random weights on every run
torch.manual_seed(42)

# Create a dataset that simulates 1000 customers
features, labels = make_classification(
    n_samples=1000,  # 1000 rows: one row per customer
    n_features=3,  # 3 columns per customer, matching the model's 3 inputs
    n_informative=3,  # all 3 features actually help to predict the label
    n_redundant=0,  # no feature is just a mix of the other features
    random_state=42,  # scikit-learn's own seed, so the same data is created every run
)

# features: one row per customer, one column per feature -> (1000, 3)
# labels: one 0 or 1 per customer -> (1000,)
print("features shape:", features.shape)
print(features[:5])  # show the first five rows of the features
print("labels shape:", labels.shape)

## 3. Splitting into Training, Validation and Test Sets

A model must be checked on customers it has **never seen**. Otherwise we cannot tell whether it learned real patterns or just memorised the training rows. So the 1000 rows are split into three sets:

```
features, labels (1000 rows)
 ├── 70% ──► train_features,   train_labels    (700)   the model learns from these
 └── 30% ──► holdout_features, holdout_labels  (300)
              ├── 50% ──► val_features,  val_labels   (150)
              └── 50% ──► test_features, test_labels  (150)
```

| Set | Rows | Share | Job |
|---|---|---|---|
| Training | 700 | 70% | The model learns from these: its weights are updated on them. |
| Validation | 150 | 15% | Checked during training, to tune settings (such as the number of epochs) and to spot overfitting. |
| Test | 150 | 15% | Used **once**, at the end, for the final honest score. |

- **Why two calls?** `train_test_split` only splits data into two parts. The first call keeps 70% for training and puts the other 30% in `holdout_features` (held out for later). The second call cuts `holdout_features` in half: 15% for validation and 15% for test.
- **`stratify=y`** keeps the same 0/1 ratio in every set: training has 351 zeros and 349 ones, and validation and test have 75 of each.
- **`random_state=42`** gives the same split on every run.

In [ ]:
# Split into training, validation and test sets (Train: 70%, Val: 15%, Test: 15%)
# Step 1: keep 70% for training; the other 30% is held out for validation and test
train_features, holdout_features, train_labels, holdout_labels = train_test_split(
    features,
    labels,
    test_size=0.3,
    random_state=42,
    stratify=labels,  # keep the same 0/1 ratio in each set
)

# Step 2: cut the held-out 30% in half: 15% for validation, 15% for test
val_features, test_features, val_labels, test_labels = train_test_split(
    holdout_features,
    holdout_labels,
    test_size=0.5,
    random_state=42,
    stratify=holdout_labels,
)

## 4. Feature Scaling with `StandardScaler`

Features often have very different ranges. In real data, age might be 18–80 while yearly spend might be 0–200,000. A feature with big numbers would dominate every weighted sum `x @ Wᵀ + b`, and training would be slow and unstable.

`StandardScaler` rescales **each column** to a mean of 0 and a standard deviation of 1:

$$z = \frac{x - \text{mean}}{\text{std}}$$

After scaling, no feature dominates just because its numbers are bigger. Our generated features are already close to this scale, but scaling is a standard step, and real data almost always needs it.

4.1 explains why the scaler is fitted on the training set only.

In [ ]:
feature_scaler = StandardScaler()

# fit on the training set only, then apply the same scaling to every set
train_features_scaled = feature_scaler.fit_transform(train_features)
val_features_scaled = feature_scaler.transform(val_features)
test_features_scaled = feature_scaler.transform(test_features)

### 4.1 ML Guide: `fit_transform` vs `transform`

![StandardScaler: fit once on the training set, transform every set](images/scaler_fit_transform.svg)

| Call | Used on | What it does |
|---|---|---|
| `feature_scaler.fit_transform(train_features)` | Training set | **Learns** each column's mean and std, **then** rescales it: `(x − mean) / std` |
| `feature_scaler.transform(val_features)` | Validation set | Rescales with the **training** mean and std. Learns nothing. |
| `feature_scaler.transform(test_features)` | Test set | Same as above |

**Why not use `fit_transform` on all three?**

1. **Data leakage.** The validation and test sets stand in for new customers the model has never seen. If the scaler learns from them, their information leaks into the pipeline, and the scores measured on them look better than the model really is.
2. **One rule for every set.** The model learns from numbers scaled by the training rule. If each set had its own mean and std, the same raw value would become a different scaled number in each set.
3. **Real-world use.** After deployment, customers arrive one at a time. A single row has no useful mean or std, so it can only be scaled with the saved training numbers: `feature_scaler.transform(new_customer)`.

**What you will see:** the scaled training set has a mean of exactly 0 and a std of exactly 1 per column. The validation and test sets are close but not exact (the test set's means are about `-0.10, 0.16, -0.09`). That is expected, because they were scaled with the training numbers, not their own.

> **Rule of thumb:** split first, then `fit` only on the training data and `transform` everything else. The same rule applies to every preprocessing step that learns from data, such as encoders, imputers and PCA.

## 5. Converting the Data to PyTorch Tensors

`train_test_split` and `StandardScaler` return **NumPy arrays**, but a PyTorch model only works with **tensors**. The cell below converts all six arrays: the features and labels of the training, validation and test sets.

In [ ]:
# Convert the NumPy arrays to PyTorch tensors, because a PyTorch model only accepts tensors.
# dtype=torch.float32: the model's weights are float32, so the inputs must be float32 too.
# .view(-1, 1) reshapes the labels from (n_samples,) to (n_samples, 1),
# the same shape as the model's output. -1 means "work out this size from the data".

# Training set: used to update the weights
train_features_tensor = torch.tensor(train_features_scaled, dtype=torch.float32)
train_labels_tensor = torch.tensor(train_labels, dtype=torch.float32).view(-1, 1)

# Validation set: used to check the model during training
val_features_tensor = torch.tensor(val_features_scaled, dtype=torch.float32)
val_labels_tensor = torch.tensor(val_labels, dtype=torch.float32).view(-1, 1)

# Test set: used once, at the end, to measure the final model
test_features_tensor = torch.tensor(test_features_scaled, dtype=torch.float32)
test_labels_tensor = torch.tensor(test_labels, dtype=torch.float32).view(-1, 1)

print("train_features_tensor shape:", train_features_tensor.shape)
print("train_labels_tensor shape:", train_labels_tensor.shape)
print("val_features_tensor shape:", val_features_tensor.shape)
print("val_labels_tensor shape:", val_labels_tensor.shape)
print("test_features_tensor shape:", test_features_tensor.shape)
print("test_labels_tensor shape:", test_labels_tensor.shape)

### 5.1 What each conversion step does

![Converting NumPy arrays to PyTorch tensors](images/tensor_conversion.svg)

Each part of the conversion prevents a real error. These are the messages PyTorch gives when a step is skipped:

| Step | Why it is needed | Error without it |
|---|---|---|
| `torch.tensor(...)` | PyTorch layers only accept tensors | `TypeError: linear(): argument 'input' (position 1) must be Tensor, not numpy.ndarray` |
| `dtype=torch.float32` on the features | NumPy gives **float64**, but the weights of `nn.Linear` are **float32**. Both sides of a matrix multiplication must have the same type. | `RuntimeError: mat1 and mat2 must have the same dtype, but got Double and Float` |
| `dtype=torch.float32` on the labels | `make_classification` gives **int64** labels (0 and 1). The loss for a yes/no prediction (`nn.BCEWithLogitsLoss`) needs float labels. | `RuntimeError: result type Float can't be cast to the desired output type Long` |
| `.view(-1, 1)` on the labels | The model returns one score per customer, shape `(batch, 1)`. The loss compares each score with its label, so the labels need the same shape. | `ValueError: Target size (torch.Size([700])) must be the same as input size (torch.Size([700, 1]))` |

**What `-1` means in `.view(-1, 1)`:** "work this size out for me". The `1` fixes the number of columns, so PyTorch computes the rows: 700 values ÷ 1 column = **700 rows**. The same line works for the 150-row validation and test sets without changing the number.

### 5.2 The resulting tensors

| Tensor | Shape | Meaning |
|---|---|---|
| `train_features_tensor` | `[700, 3]` | 700 customers × 3 features, used to update the weights |
| `train_labels_tensor` | `[700, 1]` | one label (0. or 1.) per training customer |
| `val_features_tensor`, `val_labels_tensor` | `[150, 3]`, `[150, 1]` | checks the model during training |
| `test_features_tensor`, `test_labels_tensor` | `[150, 3]`, `[150, 1]` | measures the final model, once, at the end |

## 6. Mini-batches with `TensorDataset` and `DataLoader`

The model does not see all 700 training rows at once. It learns from small groups of rows called **mini-batches**. Two PyTorch tools prepare them: `TensorDataset` pairs each customer with their label, and `DataLoader` cuts those pairs into batches.

In [ ]:
# Data loaders: feed the training data to the model in mini-batches

# TensorDataset pairs each row of features with its label,
# so train_dataset[i] gives (train_features_tensor[i], train_labels_tensor[i])
train_dataset = TensorDataset(train_features_tensor, train_labels_tensor)

# DataLoader cuts the 700 training rows into mini-batches of 32 rows
# and hands them to the training loop one batch at a time.
# shuffle=True: the rows are shuffled again at the start of every epoch,
# so the batches are different each time.
#
# Why 32? Batch sizes are usually powers of 2: 16, 32, 64, 128, 256 ...
#   - GPUs work in groups of threads whose sizes are powers of 2 (an NVIDIA GPU
#     runs threads in groups of 32), so a batch of that size divides evenly
#     between them and no thread sits idle.
#   - Memory is also laid out in power-of-2 blocks, so the data lines up neatly
#     and less memory and work are wasted.
#   - It is a common convention, so results are easy to compare with others.
# It is a habit, not a rule: any size works (e.g. batch_size=50). On a CPU, like
# here, the speed difference is usually too small to notice.
# Choosing the size: smaller batches (16) -> more updates per epoch, but noisier;
# larger batches (128) -> smoother updates, but more memory and fewer updates.
train_loader = DataLoader(train_dataset, batch_size=32, shuffle=True)

print("Rows in the dataset:", len(train_dataset))
print("Batches per epoch:", len(train_loader))  # 700 / 32 = 21.9 -> 22 batches

# Look at the first mini-batch only (list(train_loader) would print all 22 batches)
first_batch_features, first_batch_labels = next(iter(train_loader))
print("first_batch_features shape:", first_batch_features.shape)  # 32 customers x 3
print("first_batch_labels shape:", first_batch_labels.shape)  # 32 labels, one each

### 6.1 How the batches are made

![TensorDataset and DataLoader mini-batches](images/dataloader_batches.svg)

| Code | What it does |
|---|---|
| `TensorDataset(train_features_tensor, train_labels_tensor)` | Pairs row `i` of the features with label `i`. `train_dataset[0]` gives `(tensor([ 1.4325, -1.5463,  1.2566]), tensor([1.]))`: one customer and their label. |
| `DataLoader(..., batch_size=32)` | Cuts the 700 rows into batches of 32. 700 ÷ 32 = 21.9, so there are **22 batches**: 21 full batches of 32 rows and one last batch with the 28 rows left over. |
| `shuffle=True` | Mixes the rows at the start of every epoch, so each batch holds different customers each time. |
| `next(iter(train_loader))` | Takes just the first batch, so you can check its shape. `print(list(train_loader))` would print all 22 batches (700 rows). |

### 6.2 Batch, iteration and epoch

| Word | Meaning in this notebook |
|---|---|
| **Batch** | 32 rows that go through the model together |
| **Iteration** | one batch in → loss → one weight update |
| **Epoch** | one full pass over all 700 rows = **22 iterations** |

> A picture and a memory aid for these words, with a short self-test, are in **9.2**.

### 6.3 Why mini-batches instead of all 700 rows at once?

1. **More learning per epoch:** the weights are updated 22 times per epoch instead of once.
2. **Less memory:** only 32 rows are processed at a time. This matters for real datasets that are too big to fit in memory at once.
3. **Useful randomness:** each batch gives a slightly different estimate of the error. This small "noise" can help the model find weights that work well on new data, not just on the training rows.

### 6.4 Why `shuffle=True`?

Without it, every epoch would cut the rows into exactly the same batches, in the same order. If the rows were ever sorted (for example all label-0 customers first), whole batches would hold only one class, and the weights would swing back and forth from batch to batch. Shuffling gives every batch a fair mix, different each epoch. Only the training data is shuffled. The validation and test sets are only used to measure the model, so their order does not matter.

## 7. Creating the Model

This is the same `CustomerPredictor` as in Part 1 ([`Notebook.ipynb`](Notebook.ipynb), Section 3): 3 inputs → 8 hidden neurons with ReLU → 1 raw score. Each notebook runs in its **own kernel**, so classes and variables from Part 1 do not exist here, and the class has to be defined again.

The network is the same, but the names here are more descriptive:

| In Part 1 | Here |
|---|---|
| `self.hidden` | `self.hidden_layer` |
| `self.output` | `self.output_layer` |
| `forward(self, x)` | `forward(self, features)` |
| `logits = self.hidden(x)` | `hidden_values = self.hidden_layer(features)` |
| `x = torch.relu(logits)` | `hidden_activations = torch.relu(hidden_values)` |

![CustomerPredictor network: 3 input features, a hidden layer of 8 neurons with ReLU, and 1 output](images/customer_predictor_network.svg)

- `torch.manual_seed(42)` right before `CustomerPredictor()` gives the model the same starting weights on every run.
- `print(model)` lists the two layers. The parameter count is 3 × 8 + 8 + 8 × 1 + 1 = **41**, the same as in Part 1.

In [ ]:
# Create the class: the same network as in Part 1 (Notebook.ipynb), with more
# descriptive names. Each notebook runs in its own kernel, so it is defined again here.
class CustomerPredictor(nn.Module):
    def __init__(self):
        super().__init__()  # initialise nn.Module so the layers below get registered
        # input -> hidden: 3 input features, 8 neurons in one hidden layer
        self.hidden_layer = nn.Linear(3, 8)
        # hidden -> output: 8 neurons in, 1 output value
        self.output_layer = nn.Linear(8, 1)

    # forward pass: defines how an input flows through the network
    def forward(self, features):
        # Step 1: hidden layer applies z = x @ W.T + b -> shape (batch, 8)
        hidden_values = self.hidden_layer(features)
        # Step 2: ReLU keeps positive values and turns negative values into 0
        hidden_activations = torch.relu(hidden_values)
        # Step 3: output layer turns the 8 values into 1 raw score (logit) -> (batch, 1)
        return self.output_layer(hidden_activations)


# Fix the seed right before creating the model, so it starts with the same weights every run
torch.manual_seed(42)
model = CustomerPredictor()

print(model)  # the layers, in the order they were created
print("Trainable parameters:", sum(p.numel() for p in model.parameters()))  # 41

## 8. The Loss Function: Binary Cross-Entropy (BCE)

A **loss function** turns the model's mistakes into **one number**. Training then changes the weights to make that number smaller. Each customer's label is 0 or 1, so we use **binary cross-entropy**.

![Binary cross-entropy loss curves for true label 1 and true label 0](images/bce_loss_curve.svg)

| True label `y` | Predicted chance of 1, `p` | Loss | Meaning |
|---|---|---|---|
| 1 | 0.9 | 0.11 | Confident and right |
| 1 | 0.5 | 0.69 | A 50/50 guess |
| 1 | 0.1 | 2.30 | Confident and wrong |
| 0 | 0.1 | 0.11 | Confident and right |

**Why `nn.BCEWithLogitsLoss` and not `nn.BCELoss`?** The model returns a raw score (a logit), not a probability (Part 1, 3.5). `BCEWithLogitsLoss` applies the sigmoid itself and then the BCE, in one numerically stable step. `nn.BCELoss` would need a sigmoid inside the model first, and it can lose precision when the scores are very large or very small.

In [ ]:
# BCE (Binary Cross Entropy) loss function: measures how far the model's predictions are from the true labels.
# BCEWithLogitsLoss does two steps in one:
#   1. sigmoid turns each raw score (logit) into a probability p between 0 and 1
#   2. BCE compares p with the true label y (0 or 1):
#        loss = -[y * log(p) + (1 - y) * log(1 - p)]
#      confident and right -> loss close to 0; confident and wrong -> large loss
# Combining them is more numerically stable than nn.Sigmoid() followed by nn.BCELoss(),
# which is why the model's output layer has no sigmoid.
loss_function = nn.BCEWithLogitsLoss()

# optim.AdamW is a popular optimiser that updates the model's weights to reduce the loss.
# model.parameters() gives the optimiser access to the model's weights, so it can update them.
# lr = 1e-3 is a common starting learning rate for AdamW, but it can be tuned.
# weight_decay=1e-4 is a common starting value for AdamW, but it can be tuned.
optimiser = optim.AdamW(model.parameters(), lr=1e-3, weight_decay=1e-4)

## 9. Training the Model

Training repeats one small cycle for every mini-batch: **predict → measure the loss → find the gradients → update the weights**. One pass over all 22 batches is one **epoch**, so 50 epochs give 50 × 22 = **1100 weight updates**.

| Step | Code | What it does |
|---|---|---|
| 1 | `optimiser.zero_grad()` | Clears the gradients left from the previous batch. PyTorch **adds** new gradients to old ones, so they must be reset first. |
| 2 | `batch_logits = model(batch_features)` | Forward pass: one raw score per customer. |
| 3 | `batch_loss = loss_function(batch_logits, batch_labels)` | How wrong the scores are, averaged over the batch. |
| 4 | `batch_loss.backward()` | Backward pass: autograd computes the gradient of the loss for all 41 weights and biases. |
| 5 | `optimiser.step()` | AdamW moves each weight a small step in the direction that lowers the loss. |

The cell prints three kinds of logs:
- **Batch logs** (epochs 1 and 50 only): the loss of each of the 22 batches. Printing every batch of every epoch would be 1100 lines. Change `EPOCHS_WITH_BATCH_LOGS` to see other epochs.
- **Epoch logs** (every epoch): the average loss over the 22 batches, and the share of the 700 training customers predicted correctly. The accuracy is measured during the epoch, while the weights are still changing.
- **Validation logs** (every epoch): after each epoch the model is checked on the 150 validation customers. When the validation loss is the lowest so far, the weights are saved to `customer_predictor.pt` (`BEST_MODEL_PATH`) and the cell prints `Validation loss improved ...`. The code comments explain what `model.eval()` and `torch.no_grad()` do.

In [ ]:
# Re-running this cell continues training the same model from where it stopped.
# To start again from the random starting weights, re-run sections 7 and 8 first.
num_epochs = 50
train_loss_history = []  # average training loss of each epoch, to plot later
val_loss_history = []  # validation loss after each epoch, to plot later
BEST_MODEL_PATH = "customer_predictor.pt"  # file where the best weights are saved
best_val_loss = float("inf")  # start with the worst possible validation loss

# Printing all 50 x 22 = 1100 batches would flood the output, so every batch is
# printed only in these epochs. Every epoch still prints one summary line.
EPOCHS_WITH_BATCH_LOGS = {1, num_epochs}

print(
    f"Starting training: {num_epochs} epochs x {len(train_loader)} batches"
    f" = {num_epochs * len(train_loader)} weight updates"
)

for epoch_number in range(1, num_epochs + 1):
    # Training mode: the opposite of model.eval() below, which explains what changes.
    # This model has no layers that depend on the mode, so nothing changes here,
    # but it is a good habit in case such layers are added later.
    model.train()
    epoch_loss_sum = 0.0  # sum of the mini-batch losses, averaged at the end
    correct_predictions = 0  # customers predicted correctly in this epoch
    print_batch_logs = epoch_number in EPOCHS_WITH_BATCH_LOGS

    if print_batch_logs:
        print(f"\nEpoch {epoch_number}/{num_epochs}: every batch")

    # enumerate(..., 1) numbers the batches from 1, for the logs
    for batch_number, (batch_features, batch_labels) in enumerate(train_loader, 1):
        # clear the gradients from the previous mini-batch
        optimiser.zero_grad()
        # forward pass: one raw score (logit) per customer, shape (batch, 1)
        batch_logits = model(batch_features)
        # how wrong the predictions are, averaged over this mini-batch
        batch_loss = loss_function(batch_logits, batch_labels)
        # backward pass: the gradient of the loss for every weight and bias
        batch_loss.backward()
        # update the weights and biases using those gradients
        optimiser.step()

        # .item() turns the 1-value loss tensor into a plain Python float
        epoch_loss_sum += batch_loss.item()
        # a raw score above 0 means sigmoid(score) > 0.5, so the prediction is label 1
        batch_predictions = (batch_logits > 0).float()
        correct_predictions += (batch_predictions == batch_labels).sum().item()

        if print_batch_logs:
            print(
                f"  batch {batch_number:2d}/{len(train_loader)}"
                f" | rows {len(batch_features):2d} | loss {batch_loss.item():.4f}"
            )

    # epoch summary: average loss over the 22 batches, share of the 700 customers right
    avg_train_loss = epoch_loss_sum / len(train_loader)
    train_loss_history.append(avg_train_loss)  # store it for plotting later
    train_accuracy = correct_predictions / len(train_dataset)

    # print(
    #     f"Epoch {epoch_number:2d}/{num_epochs} | train loss {avg_train_loss:.4f}"
    #     f" | train accuracy {train_accuracy:.1%}"
    # )

    # Evaluation mode. model.eval() changes how two kinds of layers work:
    #   - Dropout: in training mode it switches off a random share of the values
    #     (for example 20%) in every forward pass, and scales the rest up to make
    #     up for them, so the model cannot rely on any single neuron. In eval mode
    #     it switches nothing off, so every prediction uses the whole network and
    #     gives the same answer every time.
    #   - BatchNorm: in training mode it normalises each value with the mean and
    #     variance of the current batch, and keeps a running average of them.
    #     In eval mode it uses those saved running averages instead, so a
    #     prediction does not depend on which other customers are in the batch,
    #     and it also works for a single customer.
    # This model has neither layer, so its outputs are the same in both modes.
    model.eval()

    # torch.no_grad(): in a normal forward pass, autograd records every operation
    # and keeps the intermediate results, so that backward() can compute gradients.
    # Validation never calls backward(), so that record is not needed. Turning it
    # off saves memory and time. no_grad() is about gradients; eval() is about how
    # layers behave. Validation code normally uses both.
    with torch.no_grad():
        val_logits = model(val_features_tensor)  # one raw score per customer
        val_loss = loss_function(val_logits, val_labels_tensor).item()
    val_loss_history.append(val_loss)  # store it for plotting later

    # Save the weights whenever the model does better on unseen (validation) data
    if val_loss < best_val_loss:
        best_val_loss = val_loss
        torch.save(model.state_dict(), BEST_MODEL_PATH)
        print(f"Validation loss improved to {best_val_loss:.4f}, model saved.")

    # Every 10 epochs: a progress line with the train, validation and best losses
    if epoch_number % 10 == 0:
        print(
            f"Epoch {epoch_number}/{num_epochs} | Train loss: {avg_train_loss:.4f}"
            f" | Val loss: {val_loss:.4f} | Best val loss: {best_val_loss:.4f}"
        )

In [ ]:
# Training summary: how far the average loss came down
lowest_loss_epoch = train_loss_history.index(min(train_loss_history)) + 1
print(f"First epoch loss: {train_loss_history[0]:.4f}")
print(f"Last epoch loss:  {train_loss_history[-1]:.4f}")
print(f"Lowest loss:      {min(train_loss_history):.4f} (epoch {lowest_loss_epoch})")
print(f"Final train accuracy: {train_accuracy:.1%}")
print(f"Best validation loss: {best_val_loss:.4f} (saved to {BEST_MODEL_PATH})")

### 9.1 Reading the logs

| What to look at | In the logs | What it tells you |
|---|---|---|
| Batch losses in epoch 1 | 0.63 – 0.69 | Close to 0.693, the loss of a 50/50 guess (Section 8): the model starts out guessing. |
| Batch losses in epoch 50 | 0.07 – 0.34 | Much lower, but they still jump around. Each batch holds different customers, and some are easier than others. That is why the **epoch average** is the number to follow. |
| Epoch loss | 0.6691 → 0.1999 | Falls quickly at first, then flattens out. The model is still improving at epoch 50, but only slowly. |
| Train accuracy | 50.7% → 91.7% | Jumps between epochs 4 and 7 (62.4% → 90.0%), then levels off at about 91–92%. |
| Last batch | `rows 28` | 700 = 21 × 32 + 28, so the last batch of every epoch is smaller. |
| Validation loss | 0.6627 → 0.1470, better in every epoch | The model also improves on 150 customers it never trains on, so it is learning real patterns, not memorising. It improved in all 50 epochs, so the saved model is the one from epoch 50. |
| Validation vs train loss | 0.1470 vs 0.1999 at epoch 50 | Validation is lower here, and that is normal. The training loss is averaged over the epoch while the weights are still improving; the validation loss is measured after the epoch, with the final weights. |

> **Training numbers** show how well the model fits the customers it learned from. **Validation numbers** show how it does on customers it has never seen, and they decide which weights are saved. The **test set** is still untouched: it is used once, at the very end.

### 9.2 Memory aid: batch vs epoch

![Batch vs epoch: each box is one batch of 32 customers, each row is one epoch of all 700 customers, and 50 rows make 1100 updates](images/epoch_batch_grid.svg)

**Think of studying a question bank with 700 practice questions:**

| Training word | Studying analogy | In this notebook |
|---|---|---|
| **Batch** | You do 32 questions, then check your answers. | 32 customers go through the model together. |
| **Iteration** (one update) | After checking, you fix what you got wrong. | `optimiser.step()` updates the 41 weights once. |
| **Epoch** | You have worked through all 700 questions once. | All 22 batches are done: every customer was seen once. |
| **50 epochs** | You go through the whole question bank 50 times, in a new order each time. | `num_epochs = 50`, and `shuffle=True` gives new batches each epoch. |

**The numbers to remember:**

```
batches per epoch = rows ÷ batch_size, rounded up   = 700 ÷ 32 = 21.9 → 22
total updates     = epochs × batches per epoch      = 50 × 22         = 1100
customers seen per epoch = all of them, once        = 700
```

> **In one line:** a batch is a **piece** of the data, an epoch is **all** of the data once. The weights change after every **batch**, not after every epoch.

**Test yourself:**

1. With `batch_size=64`, how many batches are in one epoch?
2. With `batch_size=32` and `num_epochs = 20`, how many times are the weights updated?
3. During one epoch, how many times does each customer go through the model?

<details>
<summary>Show the answers</summary>

1. 700 ÷ 64 = 10.9 → **11 batches** (10 batches of 64, and a last one of 60).
2. 20 × 22 = **440 updates**.
3. **Once.** That is what one epoch means.

</details>